# Assign morphotopes to the tree

Assign morphotopes and noise from the extension countries to the level 7 of the existing tree to retain consistency of the taxonomy.

In [2]:
import geopandas as gpd
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from libpysal.graph import read_parquet
import numpy as np

In [7]:
v = "v10"
v_ext = "v10_ext2"

model_params = "_post_processing_v1"

clusters_dir = "/data/uscuni-ulce/processed_data/clusters/"
chars_dir = "/data/uscuni-ulce/processed_data/chars/"
regions_datadir = "/data/uscuni-ulce/"
graph_dir = "/data/uscuni-ulce/processed_data/neigh_graphs/"

In [4]:
country = "extension_2"

region_hulls = gpd.read_parquet(
    regions_datadir + "regions/" + f"{country}_regions_hull.parquet"
)

In [5]:
cdata = pd.read_parquet(f"/data/uscuni-ulce/processed_data/clusters/cdata_{v}.pq")
cdata.index = cdata.index.astype(str)

In [6]:
def assign_to_tree(region_id, cdata):
    # read cluster data and characters
    morphotopes = pd.read_parquet(
        f"/data/uscuni-ulce/processed_data/morphotopes/tessellation_labels_morphotopes_{region_id}{model_params}.pq"
    )
    chars = pd.read_parquet(f"{chars_dir}primary_chars_{region_id}.parquet")
    chars = chars[chars.index >= 0]

    # [noise] get morphotopes and noise groups values
    is_noise = morphotopes.morphotope_label.str.contains("-1")


    # align morphotopes and cdata
    morphotope_groups = chars[~is_noise].groupby(morphotopes[~is_noise].morphotope_label).median()
    morphotope_chars = pd.read_parquet(f"/data/uscuni-ulce/processed_data/morphotopes/morph_chars_{region_id}.pq")
    morphotope_groups["limAre"] = morphotope_chars['limAre']
    morphotope_groups["limLPS"] = morphotope_chars['limLPS']
    morphotope_groups = morphotope_groups[cdata.columns]


    # combine the cluster and the region morphotope chars
    # and scale them
    all_data = pd.concat((cdata, morphotope_groups))
    scalar = StandardScaler()
    vals = scalar.fit_transform(all_data)
    all_data = pd.DataFrame(
        vals, index=all_data.index, columns=all_data.columns
    ).fillna(0)
    
    # setup tree data
    morphotope_query_data = all_data.loc[morphotope_groups.index]
    tree_data = all_data.loc[cdata.index]
    
    ## query a knn classifier and assign predictions
    tree = KNeighborsClassifier(n_neighbors=1, n_jobs=-1)
    tree = tree.fit(X=tree_data.values, y=cdata.index.values.astype(str))
    morphotope_predictions = tree.predict(morphotope_query_data.values)

    # assign morphotope labels
    morphotopes.loc[~is_noise, 'final'] = morphotopes.loc[~is_noise, 'morphotope_label'].map(
        dict(zip(morphotope_groups.index, morphotope_predictions.astype(int)))
    )

    with_noise = morphotopes.final.copy()

    # ========== noise ==========
    tq1 = read_parquet(graph_dir + f"tessellation_graph_{region_id}.parquet")
    noise_graph = tq1.subgraph(morphotopes[is_noise].index)
    components = noise_graph.component_labels
    noise_groups = chars[is_noise].groupby(components).median()

    # [noise] approax limAre
    noise_groups["limAre"] = (
        chars[is_noise]
        .groupby(components)["sdbAre"]
        .apply(lambda x: x.sort_values(ascending=True)[-min(30, x.shape[0]) :].sum())
    )
    noise_groups = noise_groups[cdata.columns.drop("limLPS")]
    noise_groups.index = "-1_" + noise_groups.index.astype(str)
    
    all_data = pd.concat((cdata.drop("limLPS", axis=1), noise_groups))
    scalar = StandardScaler()
    vals = scalar.fit_transform(all_data)
    all_data = pd.DataFrame(
        vals, index=all_data.index, columns=all_data.columns
    ).fillna(0)
    
    # setup tree data
    query_data = all_data.loc[noise_groups.index]
    tree_data = all_data.loc[cdata.index]
    
    ## query a knn classifier and assign predictions
    tree = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    tree = tree.fit(X=tree_data.values, y=cdata.index.values)
    predictions = tree.predict(query_data.values)
    
    # map noise group clusters to building ids
    bid_to_ngroup = noise_graph.component_labels
    bid_to_prediction = bid_to_ngroup.map(
        pd.Series(
            predictions, query_data.index.str.split("_").str[-1].astype(int)
        ).to_dict()
    )
    morphotopes.loc[bid_to_prediction.index.values, "final"] = (
        bid_to_prediction.values.astype(int)
    )
    return with_noise, morphotopes.final

In [8]:
%%time

for region_id, _ in region_hulls.iterrows():
    print(region_id)
    assigned, final_without_noise = assign_to_tree(region_id, cdata)
    clusters = gpd.read_parquet(f"{clusters_dir}clusters_{region_id}_{v_ext}.pq")
    clusters["final"] = assigned
    clusters["final_without_noise"] = final_without_noise
    clusters.to_parquet(f'{clusters_dir}clusters_{region_id}_{v}.pq')

537200
537429
537600
538559
538918
539534
540151
540231
540522
540571
540965
541233
541317
541334
541712
542424
542704
543980
544035
545280
545900
546287
546295
546972
547551
548395
548559
549031
549793
550169
550368
551751
552076
552430
552512
554050
554837
554976
555007
555264
555691
556455
556630
556860
557163
557292
558002
558163
558687
558796
558896
559120
559389
560821
561124
561708
562731
563089
563709
564548
564887
565573
566778
568913
568970
569421
571967
573642
573792
574242
574361
578560
579282
581173
581520
581841
582841
583461
583809
584303
584460
585155
586504
586590
586610
587468
588282
588351
588748
588777
589575
590332
590473
590852
591101
591741
592173
592444
592909
594166
595330
596149
596412
597702
597798
598030
598657
599278
600090
600429
602073
602074
602462
602842
602994
603046
603072
603130
603268
603549
603885
604943
605013
606799
606843
608794
610320
610586
611582
612217
613773
613845
613969
615370
618519
618938
619325
619671
620394
620420
620717
621979
622730